# PDF to YOLO to Vision Models


## Imports


In [1]:
import os, re, json, base64
from pathlib import Path

import cv2
import fitz            # PyMuPDF
import numpy as np
from portkey_ai import Portkey
from doclayout_yolo import YOLOv10
import yaml


C:\Users\78235\AppData\Roaming\Python\Python312\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
C:\Users\78235\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration


In [12]:
# ── Edit these values ──────────────────────────────────────────────────────
PDF_PATH        = "..\\..\\Sample PDFs\\Blueprint.pdf"
YOLO_WEIGHTS    = "..\\..\\Model\\doclayout_yolo_docstructbench_imgsz1024.pt"
PORTKEY_API_KEY = os.environ.get("PORTKEY_API_KEY")
PORTKEY_BASE    = os.environ.get("PORTKEY_BASE_URL")
MODEL_NAME      = "gpt-5.4-mini"
OUTPUT_DIR      = "output_Blueprint"          # crops + final files go here

YOLO_IMG_SIZE   = 1024
YOLO_CONF       = 0.20
RENDER_ZOOM     = 2.0
JPEG_QUALITY    = 85
JSON_INDENT = 2

with open("prompts.yaml", "r" , encoding="utf-8") as f:
    prompts = yaml.safe_load(f)

MARKDOWN_PROMPT = prompts["MARKDOWN_PROMPT"]
READING_ORDER_PROMPT = prompts["READING_ORDER_PROMPT"]
FIGURE_PROMPTS = prompts["FIGURE_PROMPTS"]

FIGURE_ROLES = {
        "bar_chart", "line_chart", "scatter_plot", "pie_chart", "histogram",
        "heatmap", "flowchart", "diagram", "map", "photo", "illustration", "plot", "figure"
    }

## Setup


In [13]:
def setup():
    """Initialize PDF document, Portkey client, YOLO model, and output directory."""
    
    doc = fitz.open(PDF_PATH)
    client = Portkey(base_url=PORTKEY_BASE, api_key=PORTKEY_API_KEY)
    model = YOLOv10(YOLO_WEIGHTS)
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    return doc, client, model


doc, client, model = setup()


## Helpers


In [14]:
def render_page(page: fitz.Page, zoom: float = RENDER_ZOOM) -> np.ndarray:
    """
    Renders a PDF page into an image array.

    Parameters:
        page (fitz.Page): The PDF page object.

    Returns:
        np.ndarray: The rendered page as an image array.
    """

    mat    = fitz.Matrix(zoom, zoom)
    pix    = page.get_pixmap(matrix=mat, alpha=False)
    img    = np.frombuffer(pix.samples, np.uint8).reshape(pix.height, pix.width, pix.n)
    conv   = cv2.COLOR_RGBA2BGR if pix.n == 4 else cv2.COLOR_RGB2BGR
    return cv2.cvtColor(img, conv)


def to_data_url(img: np.ndarray, quality: int = JPEG_QUALITY) -> str:
    """
    Converts an image array to a base64-encoded data URL.

    Parameters:
        img (np.ndarray): Image array.

    Returns:
        str: Base64-encoded data URL representation of the image.
    """

    ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, quality])
    if not ok:
        raise RuntimeError("JPEG encode failed")
    return "data:image/jpeg;base64," + base64.b64encode(buf).decode()


def save_image(img: np.ndarray, path: str) -> None:
    """
    Saves an image array to disk.

    Parameters:
        img (np.ndarray): Image array to save.
        path (str): File path where the image will be saved.

    Returns:
        None
    """

    cv2.imwrite(path, img)


def text_item(text: str) -> dict:
    """
    Creates a text content item for the Responses API.

    Parameters:
        text (str): Input text.

    Returns:
        dict: Formatted text content item.
    """

    return {"type": "input_text", "text": text}


def image_item(img: np.ndarray) -> dict:
    """
    Creates an image content item for the Responses API.

    Parameters:
        img (np.ndarray): Image array.

    Returns:
        dict: Formatted image content item with data URL.
    """

    return {"type": "input_image", "image_url": to_data_url(img)}


def create_response(content: list[dict]) -> str:
    """
    Sends content to the Responses API and retrieves the model output.

    Parameters:
        content (list[dict]): Input content for the model.

    Returns:
        str: Model-generated response text.
    """
    
    response = client.responses.create(
        model=MODEL_NAME,
        input=[{"role": "user", "content": content}],
    )
    return response.output_text.strip()

## YOLO


In [15]:
VISUAL_LABELS = {"figure", "table", "figure_caption", "table_caption",
                 "isolate_formula", "formula_caption"}

def get_yolo_outputs(page_img: np.ndarray, page_num: int) -> list[dict]:
    """
    Runs YOLO model inference on a page image to detect layout elements.

    Parameters:
        page_img (np.ndarray): Image of the PDF page.

    Returns:
        list[dict]: Detected bounding boxes and associated metadata.
    """

    Height, Width = page_img.shape[:2]

    # PRINT YOLO INPUT INFO
    
    print(f"Input to YOLO: page={page_num}, shape=({Height},{Width})")

    result = model.predict(page_img, imgsz=YOLO_IMG_SIZE, conf=YOLO_CONF, device="cpu")[0]
    boxes = result.boxes
    names = getattr(result, "names", {})

    page_dir = Path(OUTPUT_DIR) / f"page_{page_num:03d}"
    page_dir.mkdir(parents=True, exist_ok=True)

    blocks = []
    if boxes is None or len(boxes) == 0:
        print("Output from YOLO: []")
        return blocks

    for i in range(len(boxes)):
        x1, y1, x2, y2 = boxes.xyxy[i].cpu().numpy().astype(int).tolist()
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(x2, Width), min(y2, Height)
        if x2 <= x1 or y2 <= y1:
            continue

        cls_id = int(boxes.cls[i].item())
        label = names.get(cls_id, f"class_{cls_id}") if isinstance(names, dict) else str(cls_id)
        conf = float(boxes.conf[i].item())
        crop = page_img[y1:y2, x1:x2].copy()

        crop_filename = f"block_{i+1:02d}_{label}.jpg"
        crop_path = str(page_dir / crop_filename)
        save_image(crop, crop_path)

        blocks.append({
            "idx": i + 1,
            "page": page_num,
            "label": label,
            "conf": round(conf, 3),
            "bbox": (x1, y1, x2, y2),
            "crop": crop,
            "crop_path": crop_path,
        })

    blocks.sort(key=lambda b: (b["bbox"][1], b["bbox"][0]))

    # ---- Output formatting (ROW FORMAT) ----
    yolo_output = [
        f"idx={b['idx']} label={b['label']} conf={b['conf']} bbox={list(b['bbox'])}"
        for b in blocks
    ]

    print("\nOutput from YOLO:\n")
    for row in yolo_output:
        print(row)

    
    return blocks

## Vision Model 1


In [16]:
def get_reading_order(page_img: np.ndarray, blocks: list[dict], page_num: int) -> list[dict]:
    """
    Generates reading order for detected layout blocks using OpenAI.

    Parameters:
        page_img (np.ndarray): Image of the PDF page.
        block_summary (list[dict]): Summary of detected blocks.
        page_num (int): Page number.

    Returns:
        list[dict]: Reading order with block indices and roles.
    """
     
    block_summary = []
    for b in blocks:
        block_summary.append({
            "idx": b["idx"],
            "label": b["label"],
            "conf": b["conf"],
            "bbox": list(b["bbox"]),
        })


    # PRINT INPUT TO VISION MODEL 1

    H, W = page_img.shape[:2]
    print(f"\nInput to Vision Model 1: page={page_num}, shape=({H},{W}), blocks={len(block_summary)}\n")
    for row in block_summary:
        print(
            f"idx={row['idx']} "
            f"label={row['label']} "
            f"conf={row['conf']} "
            f"bbox={row['bbox']}"
        )

    content = [
        text_item(READING_ORDER_PROMPT),
        text_item(f"Page {page_num} blocks:\n" + json.dumps(block_summary, indent=JSON_INDENT)),
        image_item(page_img),
    ]

    raw = create_response(content)

    try:
        clean = re.sub(r"^```[a-z]*\n?|\n?```$", "", raw.strip(), flags=re.MULTILINE)
        reading_order = json.loads(clean)
    except json.JSONDecodeError:
        reading_order = [
            {
                "order": i + 1,
                "idx": b["idx"],
                "label": b["label"],
                "bbox": list(b["bbox"]),
                "role": "body",
                "note": "",
            }
            for i, b in enumerate(blocks)
        ]


    # SAVE READING ORDER JSON


    ro_path = Path(OUTPUT_DIR) / f"page_{page_num:03d}" / "reading_order.json"
    ro_path.parent.mkdir(parents=True, exist_ok=True)
    with open(ro_path, "w", encoding="utf-8") as f:
        json.dump(reading_order, f, indent=JSON_INDENT)


    # PRINT OUTPUT FROM VISION MODEL 1

    print("\nOutput from Vision Model 1:\n")
    for row in reading_order:
        print(
            f"order={row.get('order')} "
            f"idx={row.get('idx')} "
            f"label={row.get('label')} "
            f"bbox={row.get('bbox')} "
            f"role={row.get('role')} "
            f"note={row.get('note')}"
        )

    return reading_order

## Vision Model 2


In [17]:
def generate_markdown(page_img: np.ndarray,blocks: list[dict],reading_order: list[dict],page_num: int) -> tuple[str, list[dict]]:
    """
    Generates markdown content for a page using reading order and visual context.

    Parameters:
        page_img (np.ndarray): Image of the PDF page.
        reading_order (list[dict]): Ordered layout blocks with metadata.
        block_by_idx (dict): Mapping of block indices to block data.
        page_num (int): Page number.

    Returns:
        str: Generated markdown content.
    """

    block_by_idx = {b["idx"]: b for b in blocks}
    ro_clean = [{k: v for k, v in item.items()} for item in reading_order]


    # PRINT INPUT TO VISION MODEL 2

    H, W = page_img.shape[:2]
    print(f"\nInput to Vision Model 2: page={page_num}, shape=({H},{W}), reading_order_items={len(ro_clean)}\n")
    for item in ro_clean:
        print(
            f"order={item.get('order')} "
            f"idx={item.get('idx')} "
            f"label={item.get('label')} "
            f"bbox={item.get('bbox')} "
            f"role={item.get('role')} "
            f"note={item.get('note')}"
        )

    content = [
        text_item(MARKDOWN_PROMPT),
        text_item(f"Page {page_num}"),
        text_item("Reading order JSON:\n" + json.dumps(ro_clean, indent=JSON_INDENT)),
        image_item(page_img),
    ]

    for item in reading_order:
        if item.get("role") == "skip":
            continue

        blk = block_by_idx.get(item["idx"])
        if blk is None:
            continue

        # content.append(text_item(f"Block idx={item['idx']} label={item['label']} role={item.get('role', '')} order={item['order']}"))
        # content.append(image_item(blk["crop"]))

        role = item.get("role", "")

        # if role in FIGURE_ROLES:
        #     # Inject the role-specific summary prompt, then the figure crop
        #     figure_prompt = FIGURE_PROMPTS.get(role, FIGURE_PROMPTS["figure"])
        #     content.append(text_item(
        #         f"Block idx={item['idx']} label={item['label']} role={role} order={item['order']}\n"
        #         f"{figure_prompt}"
        #     ))
        #     content.append(image_item(blk["crop"]))


        if role in FIGURE_ROLES:
            figure_prompt = FIGURE_PROMPTS.get(role, FIGURE_PROMPTS["figure"])
            content.append(text_item(
                f"--- FIGURE BLOCK (idx={item['idx']}, role={role}, order={item['order']}) ---\n"
                f"INSTRUCTION: {figure_prompt}\n"
                f"Analyze the following image exactly as instructed above and embed the full result here:"
            ))
            content.append(image_item(blk["crop"]))
        else:
            # Non-figure block: pass label context and crop as before
            content.append(text_item(f"Block idx={item['idx']} label={item['label']} role={role} order={item['order']}"))
            content.append(image_item(blk["crop"]))

    raw = create_response(content)

    json_fence = re.compile(r"```json\s*(\[.*?\])\s*```", re.DOTALL)
    match = json_fence.search(raw)

    figures_json = []
    if match:
        try:
            figures_json = json.loads(match.group(1))
            for elem in figures_json:
                elem["page"] = page_num
        except json.JSONDecodeError:
            figures_json = []
        markdown = raw[:match.start()].strip()
    else:
        markdown = raw

    page_dir = Path(OUTPUT_DIR) / f"page_{page_num:03d}"
    with open(page_dir / "markdown.md", "w", encoding="utf-8") as f:
        f.write(markdown)
    with open(page_dir / "figures.json", "w", encoding="utf-8") as f:
        json.dump(figures_json, f, indent=JSON_INDENT, ensure_ascii=False)


    # PRINT OUTPUT FROM VISION MODEL 2

    print("\nOutput from Vision Model 2:\n")
    markdown_one_line = markdown.replace("\n", " ").strip()
    print(f"markdown={markdown_one_line}\n")

    if figures_json:
        for item in figures_json:
            row = " ".join(f"{k}={v}" for k, v in item.items())
            print(f"\nfigures_tables={row}")
    else:
        print("figures_tables=[]")

    return markdown, figures_json

## Process Page


In [18]:
def process_page(page_num: int) -> tuple[str, list[dict]]:
    """
    Processes a single PDF page end-to-end including rendering, detection, reading order, and markdown generation.

    Parameters:
        page (fitz.Page): PDF page object.
        page_num (int): Page number.

    Returns:
        dict: Outputs including reading order, markdown, and intermediate artifacts.
    """

    page = doc[page_num - 1]
    page_img = render_page(page)

    blocks = get_yolo_outputs(page_img, page_num)
    if not blocks:
        return f"\n\n## Page {page_num}\n\n[No content detected]\n", []

    reading_order = get_reading_order(page_img, blocks, page_num)
    markdown, figures_json = generate_markdown(page_img, blocks, reading_order, page_num)

    return f"\n\n## Page {page_num}\n\n{markdown}\n", figures_json


## Main Function


In [19]:
def main() -> tuple[Path, Path]:
    """
    Runs the full PDF processing pipeline and saves markdown and figure outputs.

    Parameters:
        None

    Returns:
        tuple[Path, Path]: Paths to the generated markdown file and figures JSON file.
    """
    all_page_markdowns: dict[int, str] = {}
    all_figures: list[dict] = []
    total_pages = len(doc)

    for page_num in range(1 , total_pages + 1):
        try:
            page_md, page_figs = process_page(page_num)
            all_page_markdowns[page_num] = page_md
            all_figures.extend(page_figs)
        except Exception as exc:
            print(f"Page {page_num} failed: {exc}")
            all_page_markdowns[page_num] = f"\n\n## Page {page_num}\n\n[Failed: {exc}]\n"

    full_markdown = "".join(all_page_markdowns[p] for p in range(1 , total_pages + 1))
    out = Path(OUTPUT_DIR)

    md_path = out / "document.md"
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(f"# {Path(PDF_PATH).stem}\n")
        f.write(full_markdown)

    json_path = out / "figures_tables.json"
    serialisable = [{k: v for k, v in e.items() if k != "crop"} for e in all_figures]
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(serialisable, f, indent=JSON_INDENT, ensure_ascii=False)

    return md_path, json_path

## Execution


In [20]:
md_path, json_path = main()
print(md_path)
print(json_path)

Input to YOLO: page=1, shape=(1584,1224)

0: 1024x800 2 titles, 6 abandons, 1 figure, 8857.6ms
Speed: 29.6ms preprocess, 8857.6ms inference, 4.1ms postprocess per image at shape (1, 3, 1024, 800)

Output from YOLO:

idx=8 label=abandon conf=0.552 bbox=[144, 71, 552, 198]
idx=9 label=title conf=0.245 bbox=[144, 71, 552, 198]
idx=3 label=abandon conf=0.864 bbox=[682, 143, 1082, 196]
idx=6 label=title conf=0.755 bbox=[142, 242, 430, 268]
idx=5 label=abandon conf=0.833 bbox=[977, 244, 1081, 267]
idx=1 label=figure conf=0.92 bbox=[165, 295, 1058, 1365]
idx=7 label=abandon conf=0.754 bbox=[1032, 1375, 1090, 1433]
idx=2 label=abandon conf=0.893 bbox=[965, 1454, 1064, 1552]
idx=4 label=abandon conf=0.855 bbox=[141, 1507, 476, 1528]

Input to Vision Model 1: page=1, shape=(1584,1224), blocks=9

idx=8 label=abandon conf=0.552 bbox=[144, 71, 552, 198]
idx=9 label=title conf=0.245 bbox=[144, 71, 552, 198]
idx=3 label=abandon conf=0.864 bbox=[682, 143, 1082, 196]
idx=6 label=title conf=0.755 bbox=[